# 02 - Baseline OLS (Table 1)

Estimates the full-sample safe-haven regression (Table 1) on the free-data
snapshot from `01_data`. For each currency the daily **excess
return** is regressed on five risk factors, estimated with **Newey-West (HAC)**
standard errors, `maxlags = 2`:

$$ \text{excess}_t = \alpha + \beta_1 \text{SP500}_t + \beta_2 \text{10Y}_t
   + \beta_3 \text{fxvol}_t + \beta_4 \text{TED}_t + \beta_5 \text{VIX}_t + \varepsilon_t $$

Variable construction: excess return = log FX appreciation +
interest-rate differential; SP500 and VIX as daily log changes; TED = US 3M
interbank − 3M T-bill; fx volatility = 30-day rolling volatility averaged over
the *other* three currencies. Rates enter daily-ized (annual / 360).


## Setup

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

import safe_haven as sh
from safe_haven import CURRENCIES, DATA_DIR, PAPER_DIR

## Load and construct variables

Returns and volatility are built on a **gap-free FX panel** (days with any missing exchange rate are dropped first), matching the thesis's handling of incomplete days.

The construction lives in `safe_haven.py`, so every notebook uses one definition.

In [2]:
data = sh.load_data()
excess, fx_vol = data.excess, data.fx_vol
sp500_d, ust10y_d, ted, vix_d = data.sp500_d, data.ust10y_d, data.ted, data.vix_d

## FX volatility

30-day rolling standard deviation of log returns; each currency's regressor is
the average realized volatility of the *other* three (leave-one-out).

## Per-currency regression (Newey-West, lag 2)

In [3]:
def estimate(cur):
    df = pd.concat([
        excess[cur].rename("excess"),
        sp500_d, ust10y_d, fx_vol[cur].rename("fxvol"), ted, vix_d,
    ], axis=1).dropna()
    model = smf.ols("excess ~ SP500 + UST10Y + fxvol + TED + VIX", data=df)
    return model.fit(cov_type="HAC", cov_kwds={"maxlags": 2})

results = {cur: estimate(cur) for cur in CURRENCIES}

In [4]:
# Robustness: the thesis' Newey-West lag of 2 is short for daily data with a
# 30-day rolling regressor; compare t-values against longer bandwidths.
hac = {}
for cur in CURRENCIES:
    df = pd.concat([
        excess[cur].rename("excess"),
        sp500_d, ust10y_d, fx_vol[cur].rename("fxvol"), ted, vix_d,
    ], axis=1).dropna()
    auto = int(4 * (len(df) / 100) ** (2 / 9))          # Newey-West rule of thumb
    model = smf.ols("excess ~ SP500 + UST10Y + fxvol + TED + VIX", data=df)
    hac[cur] = pd.DataFrame(
        {f"lag {lag}": model.fit(cov_type="HAC", cov_kwds={"maxlags": lag}).tvalues
         for lag in (2, auto, 30)}).drop("Intercept").round(1)
L = [r"\begin{table}[H]", r"\centering",
     r"\caption{Baseline $t$-statistics under alternative Newey-West bandwidths: "
     r"2 lags (thesis), the Newey-West (1994) rule of thumb, and 30 lags.}",
     r"\label{tab:hac}", r"\begin{tabular}{ll" + "r" * 3 + "}", r"\toprule",
     r"Currency & Regressor & " + " & ".join(hac["CHF"].columns) + r" \\", r"\midrule"]
for cur in CURRENCIES:
    for i, reg in enumerate(hac[cur].index):
        name = cur if i == 0 else ""
        L.append(f"{name} & {reg} & " + " & ".join(
            f"${hac[cur].loc[reg, c]:.1f}$" for c in hac[cur].columns) + r" \\")
    if cur != CURRENCIES[-1]:
        L.append(r"\midrule")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
PAPER_DIR.mkdir(exist_ok=True)
(PAPER_DIR / "hac_robustness.tex").write_text("\n".join(L))
pd.concat(hac, axis=1)

CHF                EUR                GBP                JPY        \
       lag 2 lag 9 lag 30 lag 2 lag 9 lag 30 lag 2 lag 9 lag 30 lag 2 lag 9   
SP500    2.5   2.6    2.4   5.5   5.5    4.8   5.3   5.3    5.3  -0.6  -0.6   
UST10Y -11.2 -10.2   -8.8  -7.1  -6.4   -5.4  -4.1  -3.7   -3.1 -16.9 -16.1   
fxvol    1.1   1.1    1.3   0.6   0.6    0.7   0.6   0.7    0.7   1.9   1.9   
TED     -0.6  -0.7   -0.8  -0.7  -0.7   -0.7  -1.7  -1.8   -2.2   0.9   1.1   
VIX      2.2   2.2    2.1   1.2   1.1    1.0   0.2   0.2    0.2   4.1   3.8   

               
       lag 30  
SP500    -0.6  
UST10Y  -14.2  
fxvol     2.1  
TED       1.2  
VIX       3.7

## Table 1 - coefficients

Coefficient with t-statistic beneath, per currency.

In [5]:
TERMS = ["Intercept", "SP500", "UST10Y", "fxvol", "TED", "VIX"]

def cell(res, term):
    return f"{res.params[term]:.3f} (t={res.tvalues[term]:.1f})"

table1 = pd.DataFrame(
    {cur: {term: cell(results[cur], term) for term in TERMS} for cur in CURRENCIES})
table1.loc["N"] = {cur: int(results[cur].nobs) for cur in CURRENCIES}
table1.loc["R2"] = {cur: round(results[cur].rsquared, 3) for cur in CURRENCIES}
table1

,CHF,EUR,GBP,JPY
Intercept,0.000 (t=1.2),0.000 (t=0.8),0.000 (t=1.5),-0.000 (t=-1.2)
SP500,0.030 (t=2.5),0.068 (t=5.5),0.085 (t=5.3),-0.009 (t=-0.6)
UST10Y,-0.022 (t=-11.2),-0.013 (t=-7.1),-0.008 (t=-4.1),-0.035 (t=-16.9)
fxvol,0.000 (t=1.1),0.000 (t=0.6),0.000 (t=0.6),0.001 (t=1.9)
TED,-0.066 (t=-0.6),-0.084 (t=-0.7),-0.215 (t=-1.7),0.119 (t=0.9)
VIX,0.004 (t=2.2),0.002 (t=1.2),0.000 (t=0.2),0.009 (t=4.1)
N,5790,5790,5790,5286
R2,0.04,0.023,0.029,0.141


## Where does the R² come from?

Re-fit each currency without UST10Y, and with UST10Y alone, to see how much of the fit hangs on the ten-year yield. For the JPY the yield on its own gives an R² of 0.128, and taking it out of the full model costs 0.098 (from 0.141 to 0.043). The two numbers differ because the regressors overlap.

In [6]:
# R^2 decomposition: full model vs. dropping UST10Y vs. UST10Y alone.
r2 = {}
for cur in CURRENCIES:
    df = pd.concat([excess[cur].rename("excess"), sp500_d, ust10y_d,
                    fx_vol[cur].rename("fxvol"), ted, vix_d], axis=1).dropna()
    full = smf.ols("excess ~ SP500 + UST10Y + fxvol + TED + VIX", data=df).fit()
    no10 = smf.ols("excess ~ SP500 + fxvol + TED + VIX", data=df).fit()
    only = smf.ols("excess ~ UST10Y", data=df).fit()
    r2[cur] = {"R2 full": full.rsquared,
               "R2 without UST10Y": no10.rsquared,
               "R2 UST10Y only": only.rsquared}
pd.DataFrame(r2).T.round(3)

,R2 full,R2 without UST10Y,R2 UST10Y only
CHF,0.040,0.003,0.038
EUR,0.023,0.008,0.008
GBP,0.029,0.024,0.001
JPY,0.141,0.043,0.128


## Timing check: noon exchange rates, end-of-day regressors

The FRED exchange rates are noon buying rates in New York. The S&P 500, the VIX and the Treasury yield are end-of-day values. A daily FX return therefore runs from noon to noon, and it overlaps with *yesterday's* afternoon in the equity and bond markets just as much as with this morning's. If that matters, yesterday's regressors should carry a coefficient of their own.

They do. The cell below adds one lag of the S&P 500, the yield change and the VIX. For each of the three it reports the same-day coefficient, the lagged one, and their sum with a Newey-West t-statistic. The sum is the better estimate of the full daily response; the same-day coefficient alone understates it.

In [7]:
LAGGED = ["SP500", "UST10Y", "VIX"]
rows, r2_lag = {}, {}
for cur in CURRENCIES:
    df = data.model_frame(cur)
    for v in LAGGED:
        df[f"{v}_lag"] = df[v].shift(1)
    df = df.dropna()
    base = smf.ols("excess ~ SP500 + UST10Y + fxvol + TED + VIX", data=df).fit()
    lag = smf.ols("excess ~ SP500 + SP500_lag + UST10Y + UST10Y_lag + fxvol + TED "
                  "+ VIX + VIX_lag", data=df).fit(cov_type="HAC", cov_kwds={"maxlags": 2})
    for v in LAGGED:
        total = lag.t_test(f"{v} + {v}_lag = 0")
        rows[(cur, v)] = {"same day": lag.params[v], "lag 1": lag.params[f"{v}_lag"],
                          "t (lag 1)": lag.tvalues[f"{v}_lag"],
                          "sum": total.effect.item(), "t (sum)": total.tvalue.item()}
    r2_lag[cur] = {"R2 baseline": base.rsquared, "R2 with lags": lag.rsquared}

lag_table = pd.DataFrame(rows).T.round(3)
lag_table.to_csv(DATA_DIR / "table1_lags.csv")
lag_table

same day  lag 1  t (lag 1)    sum  t (sum)
CHF SP500      0.034  0.055      3.861  0.089    4.332
    UST10Y    -0.022 -0.010     -4.889 -0.032  -10.562
    VIX        0.004  0.004      2.398  0.009    3.183
EUR SP500      0.074  0.083      6.028  0.157    7.868
    UST10Y    -0.013 -0.009     -4.352 -0.021   -7.809
    VIX        0.003  0.006      2.957  0.008    2.740
GBP SP500      0.092  0.092      5.888  0.184    7.990
    UST10Y    -0.008 -0.007     -3.555 -0.015   -5.555
    VIX        0.001  0.004      1.840  0.004    1.425
JPY SP500     -0.009  0.029      1.928  0.020    0.942
    UST10Y    -0.035 -0.017     -7.739 -0.052  -18.110
    VIX        0.009  0.005      2.550  0.014    4.790

In [8]:
pd.DataFrame(r2_lag).T.round(3)

,R2 baseline,R2 with lags
CHF,0.040,0.051
EUR,0.023,0.041
GBP,0.029,0.053
JPY,0.141,0.166


The lagged S&P 500 coefficient is about as large as the same-day one for CHF, EUR and GBP, and the lagged yield change is clearly significant for all four. So the baseline table understates the daily response, roughly by half for equities. Notebooks 03 to 07 keep the specification without lags, so that their results stay comparable with Table 1. The README states this as a limitation.

## Full model output (reference)

In [9]:
results["JPY"].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 excess   R-squared:                       0.141
Model:                            OLS   Adj. R-squared:                  0.140
Method:                 Least Squares   F-statistic:                     90.36
Date:                Tue, 17 Nov 2026   Prob (F-statistic):           1.68e-91
Time:                        11:59:38   Log-Likelihood:                 19786.
No. Observations:                5286   AIC:                        -3.956e+04
Df Residuals:                    5280   BIC:                        -3.952e+04
Df Model:                           5                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0001      0.000     -1.235      0.217      -0.000     8.6e-05
SP500         -0.0089      0.014     -0.634      0.526      -0.037       0.019
UST10Y        -0.0353      0.002    -16.926      0.000      -0.039      -0.031
fxvol          0.0005      0.000      1.852      0.064   -2.92e-05       0.001
TED            0.1193      0.127      0.941      0.347      -0.129       0.368
VIX            0.0085      0.002      4.088      0.000       0.004       0.013
==============================================================================
Omnibus:                      811.307   Durbin-Watson:                   2.145
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             9689.959
Skew:                           0.334   Prob(JB):                         0.00
Kurtosis:                       9.599   Cond. No.                         912.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 2 lags and without small sample correction
"""

## Persist

In [10]:
reproduced = pd.DataFrame(
    {cur: {term: results[cur].params[term] for term in TERMS} for cur in CURRENCIES})
out = DATA_DIR / "table1_reproduced.csv"
reproduced.round(4).to_csv(out)
print(f"Saved -> {out.relative_to(sh.ROOT)}")

Saved -> data/table1_reproduced.csv


## Export Table 1 to LaTeX

Write the baseline table (coefficient, significance stars, Newey-West standard
error, N, R²) to `../paper/table1.tex`, which the paper includes via
`\input{table1.tex}`. Re-running this notebook regenerates the table in the
paper, the notebook stays the single source of truth.

In [11]:
def _star(p):
    return "^{***}" if p < 0.01 else "^{**}" if p < 0.05 else "^{*}" if p < 0.10 else ""

ROWS = [("SP500", "SP500"), ("UST10Y", "UST10Y"), ("fxvol", "fxvol"),
        ("TED", "TED"), ("VIX", "VIX"), ("Intercept", "Constant")]

lines = [r"\begin{table}[H]", r"\centering",
         r"\caption{Excess-return regression: static OLS estimates. Newey-West (HAC) "
         r"standard errors in parentheses, lag 2. Significance: $^{*}p<0.1$, $^{**}p<0.05$, $^{***}p<0.01$.}",
         r"\label{tab:baseline}",
         r"\begin{tabular}{lrrrr}", r"\toprule",
         " & " + " & ".join(CURRENCIES) + r" \\", r"\midrule"]
for key, label in ROWS:
    lines.append(f"{label} & " + " & ".join(
        f"${results[c].params[key]:.3f}{_star(results[c].pvalues[key])}$" for c in CURRENCIES) + r" \\")
    lines.append(" & " + " & ".join(f"$({results[c].bse[key]:.3f})$" for c in CURRENCIES) + r" \\")
lines += [r"\midrule",
          "Observations & " + " & ".join(str(int(results[c].nobs)) for c in CURRENCIES) + r" \\",
          r"$R^2$ & " + " & ".join(f"{results[c].rsquared:.3f}" for c in CURRENCIES) + r" \\",
          r"\bottomrule", r"\end{tabular}", r"\end{table}"]

out = PAPER_DIR
out.mkdir(exist_ok=True)
(out / "table1.tex").write_text("\n".join(lines))
print("Wrote paper/table1.tex")

Wrote paper/table1.tex
